# Portfolio what-if: Gini, calibration, stability, and a pooled refit

`evaluate_segments` always scores the **pooled book** as the synthetic segment
`__overall__` / `ALL`. Q1 on that row is the same three pillars used on every
business segment:

| Pillar | Metric | Gate |
| --- | --- | --- |
| Rank order | Gini (absolute floor; the ratio to itself is 1) | `gini_floor` |
| Calibration | O/E and ECE | `oe_lo` / `oe_hi`, `ece_max` |
| Stability | latest / early vintage Gini | `vintage_gini_ratio_floor` |

When a production WoE grouping is supplied, the same row also carries
`result.grouping_summary` (IV, bin count, univariate Gini) and the production
predictor-stability table (`stability_source=production`).

`evaluate_portfolio_whatif` then asks the Q2 question of the **scorecard itself**:
if we re-estimated the same predictors on a forward holdout, would the new PD
beat the production one? The action is `REFIT` (replace the pooled scorecard),
`MONITOR` (lift is real, inputs are not stable), `RECALIBRATE`, or `KEEP_POOLED`.
`SPLIT` is never returned — there is no parent portfolio to split from.

This notebook uses the same synthetic book and gates as `tests/test_evaluate.py`.
The package targets Python 3.6+, so nothing here uses 3.7+ syntax.

In [ ]:
import os
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from scorecard_segment_eval import (
    BinningModel,
    Gates,
    decision_table,
    evaluate_portfolio_whatif,
    evaluate_segments,
    make_synthetic_book,
    recommendations,
    save_report,
)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 80)


def _find_repo_root():
    here = os.path.abspath(os.getcwd())
    for candidate in (here, os.path.dirname(here)):
        if os.path.isdir(os.path.join(candidate, "scorecard_segment_eval")):
            return candidate
    return here


REPO_ROOT = _find_repo_root()
WORKDIR = os.path.join(REPO_ROOT, "notebooks", "portfolio_whatif_output")
if not os.path.isdir(WORKDIR):
    os.makedirs(WORKDIR)

df, cols = make_synthetic_book(n=9000, seed=3)
gates = Gates(
    min_n=400,
    min_events=25,
    n_bootstrap=80,
    bootstrap_seed=0,
    holdout_frac=0.3,
)
obs = df.loc[df["obs"].eq(1)].copy()
print("rows", len(df), "observable", len(obs), "channels", sorted(df["channel"].unique().tolist()))
print("artefacts ->", WORKDIR)

## 1. Portfolio Q1 — Gini, calibration, stability, WoE grouping

Fit the production grouping on the observable book, then call `evaluate_segments`.
The first decision row is `__overall__` / `ALL`. Segment cells are unchanged.

In [ ]:
grouping = BinningModel.fit(obs[list(cols.cols_pred)], obs[cols.col_target], gates)
print("production WoE grouping")
print(grouping.iv_table().to_string(index=False))

result = evaluate_segments(df, cols, gates, grouping=grouping, n_jobs=1)
print("\nmeta overall_gini", result.meta["overall_gini"], "grouping_supplied", result.meta["grouping_supplied"])
print("\ndecision table (portfolio row first)")
print(decision_table(result).to_string(index=False))

In [ ]:
port = result.decisions.loc[result.decisions["segment_col"].eq("__overall__")].iloc[0]
print("Q1", port["q1_verdict"], "pillars", port["failed_pillars"] or "-")
print("Q2", port["q2_action"], port["q2_reason"])
print(
    "Gini %.3f  O/E %.2f  ECE %.3f  vintage Gini ratio %s"
    % (
        port["gini"],
        port["oe"],
        port["ece"],
        "-" if pd.isna(port["vintage_gini_ratio"]) else "%.2f" % port["vintage_gini_ratio"],
    )
)
print("\nWoE grouping on the observable book")
print(result.grouping_summary.to_string(index=False))
print("\nproduction predictor stability")
stab = result.stability
prod = stab.loc[stab["segment_col"].eq("__overall__")]
if "stability_source" in prod.columns:
    prod = prod.loc[prod["stability_source"].fillna("production").eq("production")]
cols_show = [
    c
    for c in (
        "feature",
        "n_vintages",
        "psi_max",
        "gini_reference",
        "gini_ratio_median",
        "sign_consistency",
        "stability_score",
        "stable",
        "stability_flags",
    )
    if c in prod.columns
]
print(prod[cols_show].to_string(index=False))

### Vintage Gini of the pooled score

The `__overall__` vintage table is the book-level Gini and O/E by decision month.
A drop in the latest / early ratio is the Q1 stability pillar.

In [ ]:
vint = result.vintage.loc[result.vintage["segment_col"].eq("__overall__")].copy()
print(vint.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(vint["vintage"], vint["gini"], marker="o", color="black")
axes[0].set_title("portfolio Gini")
axes[0].set_ylabel("Gini")
axes[1].plot(vint["vintage"], vint["oe"], marker="o", color="black")
axes[1].axhline(1.0, color="0.6", linestyle="--", linewidth=1)
axes[1].set_title("portfolio O/E")
axes[1].set_ylabel("O/E")
for ax in axes:
    ax.set_xlabel("vintage")
    ax.tick_params(axis="x", labelrotation=45)
fig.tight_layout()
path = os.path.join(WORKDIR, "portfolio_vintage.png")
fig.savefig(path, dpi=120)
print("wrote", path)
plt.close(fig)

### WoE grouping vs vintage

`BinningModel.plot_vintage_stability` draws event rate, bin share and univariate
Gini of each production grouping. Adjacent bins whose vintage event-rate Wilson
intervals overlap are flagged on the stability table.

In [ ]:
warnings.filterwarnings("ignore", category=UserWarning)
for feature in list(cols.cols_pred):
    fig, axes = grouping.plot_vintage_stability(
        obs, obs[cols.col_target], cols.col_date, feature, min_rows=40
    )
    path = os.path.join(WORKDIR, "grouping_vintage_%s.png" % feature)
    fig.savefig(path, dpi=120, bbox_inches="tight")
    print("wrote", path)
    plt.close(fig)

## 2. What-if: refit the pooled scorecard

`evaluate_portfolio_whatif` skips per-segment cells and **always** fits a
same-predictor holdout refit plus a two-parameter PD recalibration, even when
Q1 is `GOOD`. New WoE bins are compared to the production grouping.

In [ ]:
whatif = evaluate_portfolio_whatif(df, cols, gates, grouping=grouping, n_jobs=1)
print("only segment_col", sorted(set(whatif.decisions["segment_col"])))
print(whatif.decisions[
    [
        "q1_verdict",
        "failed_pillars",
        "q2_action",
        "q2_reason",
        "gini",
        "oe",
        "ece",
        "vintage_gini_ratio",
        "stability_pass",
        "stability_reason",
    ]
].to_string(index=False))
print("\nholdout comparison")
print(
    whatif.refit_comparison[
        [
            "n_holdout",
            "gini_pooled",
            "gini_refit",
            "delta_gini",
            "delta_gini_ci_low",
            "gini_recal",
            "brier_pooled",
            "brier_refit",
            "logloss_pooled",
            "logloss_refit",
            "refit_method",
        ]
    ].to_string(index=False)
)

In [ ]:
print("grouping comparison (refit bins vs production), significant first")
cmp_ = whatif.grouping_comparison.copy()
if cmp_.empty:
    print("(no grouping_comparison rows)")
else:
    cmp_ = cmp_.sort_values(["significant", "feature"], ascending=[False, True])
    show = [
        c
        for c in (
            "feature",
            "segment_bin",
            "portfolio_bins",
            "woe_segment",
            "woe_portfolio",
            "woe_delta",
            "kind",
            "significant",
            "note",
        )
        if c in cmp_.columns
    ]
    print(cmp_[show].head(20).to_string(index=False))
    print("... %d rows" % len(cmp_))

print("\nrecommendations")
print(recommendations(whatif, gates).to_string(index=False))

## 3. Saved artefacts and the report

The what-if refit writes the same artefact layout as a segment refit
(`grouping.json`, `model.pkl`, `meta.json`, `scorecard.sql`) under
`__overall__/ALL/refit/`.

In [ ]:
art_dir = os.path.join(WORKDIR, "artifacts")
written = whatif.save_artifacts(art_dir)
print("wrote %d artefact dirs" % len(written))
for path in written:
    print(" -", os.path.relpath(path, WORKDIR), sorted(os.listdir(path)))

html_path = save_report(
    whatif,
    os.path.join(WORKDIR, "portfolio_whatif_report.html"),
    gates=gates,
    title="Portfolio what-if refit",
)
condensed_path = save_report(
    whatif,
    os.path.join(WORKDIR, "portfolio_whatif_report_condensed.html"),
    fmt="condensed",
    gates=gates,
    title="Portfolio what-if refit (condensed)",
)
print("report", html_path)
print("condensed", condensed_path)